# Assignment 7 — GraphRAG Knowledge Explorer with Neo4j

**Module 6 · Advanced RAG & GraphRAG**

Basic RAG retrieves isolated chunks and cannot reason across connections. This notebook
builds a GraphRAG system that combines a **Neo4j knowledge graph** with **vector search**
to answer multi-hop questions.

Pipeline:

1. Chunk a knowledge-rich document into paragraphs
2. Extract `(entity1, relationship, entity2)` triples with an Ollama model using structured output
3. Load the triples into Neo4j with Cypher
4. `graph_retrieval(query)` — find query entities, traverse 1–2 hops
5. Fuse graph + vector results with Reciprocal Rank Fusion
6. Compare pure vector RAG vs GraphRAG on 5 multi-hop questions

### Prerequisites

```bash
pip install -U neo4j langchain-neo4j chromadb sentence-transformers \
               langchain langchain-core langchain-ollama langchain-text-splitters

ollama serve
ollama pull llama3.2
```

Start Neo4j (Community Edition locally, or a free Neo4j Aura instance) and set the
connection details in the next cell. With Docker:

```bash
docker run --name neo4j -p 7474:7474 -p 7687:7687 \
  -e NEO4J_AUTH=neo4j/password123 neo4j:5-community
```

In [1]:
# ============================================================
# 1. Configuration
# ============================================================

NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "password123"

OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_CHAT_URL = f"{OLLAMA_BASE_URL}/api/chat"

# llama3.2 follows the extraction schema reliably.
# qwen3:1.7b also works. qwen3:0.6b is too small for clean triple extraction.
EXTRACTION_MODEL = "llama3.2"
ANSWER_MODEL = "llama3.2"

DOCUMENT_PATH = "graph_document.txt"

In [4]:
# ============================================================
# 2. Imports
# ============================================================

import json
import re

import numpy as np
import requests
import chromadb

from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

print("Imports OK")

Imports OK


In [22]:
# ============================================================
# 3. Connect to Neo4j
# ============================================================

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD)
)

driver.verify_connectivity()

print("Connected to Neo4j at", NEO4J_URI)


def run_cypher(query, parameters=None):
    """Run a Cypher statement and return the records as dictionaries."""

    with driver.session() as session:
        result = session.run(query, parameters or {})
        return [record.data() for record in result]

Connected to Neo4j at bolt://localhost:7687


In [6]:
# ============================================================
# 4. Load Document and Chunk into Paragraphs
# ============================================================

with open(DOCUMENT_PATH, "r", encoding="utf-8") as file:
    document_text = file.read()

# The assignment asks for paragraph-level chunks, so split on blank lines
# instead of using a fixed character window.
paragraphs = [
    para.strip()
    for para in re.split(r"\n\s*\n", document_text)
    if para.strip()
]

chunks = paragraphs
chunk_ids = [f"chunk_{i}" for i in range(len(chunks))]

print("Words:", len(document_text.split()))
print("Paragraph chunks:", len(chunks))
print("\nFirst chunk:\n")
print(chunks[0])

Words: 519
Paragraph chunks: 31

First chunk:

Indian Technology Companies: Founders, Cities and Products


## 5. Entity & Relationship Extraction

The model is constrained with a **JSON schema** passed to Ollama's `format` parameter, so it
cannot return prose around the JSON. The schema forces every triple to carry both entity
types, which is what lets us label the nodes in Neo4j.

In [7]:
# ============================================================
# 5. Extraction Schema and Prompt
# ============================================================

ENTITY_TYPES = ["Person", "Organization", "Location", "Concept", "Product"]

RELATIONSHIP_TYPES = [
    "founded_by",
    "headquartered_in",
    "works_at",
    "acquired",
    "part_of",
    "develops",
    "studied_at",
    "competes_with",
    "related_to",
]

TRIPLE_SCHEMA = {
    "type": "object",
    "properties": {
        "triples": {
            "type": "array",
            # Bounded on purpose. With an unbounded array the model can loop
            # re-emitting variations of the same triple until it exhausts the
            # context window - see the note under extract_triples().
            "maxItems": 12,
            "items": {
                "type": "object",
                "properties": {
                    "entity1": {"type": "string"},
                    "entity1_type": {"type": "string", "enum": ENTITY_TYPES},
                    "relationship": {"type": "string", "enum": RELATIONSHIP_TYPES},
                    "entity2": {"type": "string"},
                    "entity2_type": {"type": "string", "enum": ENTITY_TYPES},
                },
                "required": [
                    "entity1",
                    "entity1_type",
                    "relationship",
                    "entity2",
                    "entity2_type",
                ],
            },
        }
    },
    "required": ["triples"],
}


def extract_triples(chunk):
    """Extract (entity1, relationship, entity2) triples from one chunk."""

    prompt = f"""Extract entities and relationships from the text as triples.

Entity types: {", ".join(ENTITY_TYPES)}
Relationship types: {", ".join(RELATIONSHIP_TYPES)}

Rules:
- Use exact entity names from the text.
- Use only the relationship types listed above.
- Only extract facts that are actually stated in the text.
- Do not invent entities.
- If the text states no relationships at all, return an empty list.

Direction matters. Read the relationship name literally, do not just copy
the order the words appear in the sentence:
- founded_by: entity1 is the ORGANIZATION, entity2 is the PERSON.
  "Sachin Bansal founded Navi" -> (Navi, founded_by, Sachin Bansal)
- acquired: entity1 is the BUYER, entity2 is the company that was bought.
  "Zomato acquired Blinkit" -> (Zomato, acquired, Blinkit)
  (Both ends are Organizations here, so entity types cannot recover the
  direction afterwards the way they can for founded_by. The relation is named
  in the active voice to match how the sentence is written, because a small
  model copies subject-verb-object order regardless of what the prompt says.)
- headquartered_in: entity1 is the ORGANIZATION, entity2 is the LOCATION.
- studied_at / works_at: entity1 is the PERSON.

Text:
{chunk}
"""

    try:
        response = requests.post(
            OLLAMA_CHAT_URL,
            json={
                "model": EXTRACTION_MODEL,
                "messages": [{"role": "user", "content": prompt}],
                "stream": False,
                "format": TRIPLE_SCHEMA,
                "options": {
                    "temperature": 0,
                    # Hard ceiling on generation. A schema that requires an
                    # array gives the model no way to say "nothing here", so on
                    # a chunk with no facts - the document title, for instance -
                    # it loops emitting junk triples. Unbounded, that ran to
                    # 13,000+ tokens and killed the whole loop on a timeout.
                    "num_predict": 512,
                },
            },
            # Short enough that a stuck chunk fails fast instead of stalling
            # the notebook for ten minutes.
            timeout=180,
        )

        response.raise_for_status()

    except requests.RequestException as error:
        # One bad chunk must not discard the other thirty.
        print(f"  request failed ({type(error).__name__}), skipping chunk")
        return []

    content = response.json()["message"]["content"]

    try:
        return json.loads(content).get("triples", [])
    except json.JSONDecodeError:
        print("  Could not parse response, skipping chunk")
        return []


# Quick check on a single chunk
for triple in extract_triples(chunks[3]):
    print(
        f"({triple['entity1']}) -[{triple['relationship']}]-> ({triple['entity2']})"
    )

(Wipro) -[headquartered_in]-> (Bengaluru)
(Wipro) -[founded_by]-> (Azim Premji)
(Wipro) -[develops]-> (Wipro Cloud)


In [8]:
# ============================================================
# 6. Extract Triples from Every Chunk
# ============================================================

all_triples = []
empty_chunks = 0

for i, chunk in enumerate(chunks):

    print(f"Extracting chunk {i + 1}/{len(chunks)}", flush=True)

    triples = extract_triples(chunk)

    if not triples:
        empty_chunks += 1

    for triple in triples:
        triple["source_chunk"] = chunk_ids[i]

    all_triples.extend(triples)

print("\nTotal triples extracted:", len(all_triples))
print("Chunks that produced no triples:", empty_chunks, "/", len(chunks))

# A graph built from zero triples still "works" - every query just returns
# nothing and GraphRAG quietly degrades to plain vector search. Fail loudly.
assert all_triples, "No triples extracted - check that Ollama is running"

Extracting chunk 1/31
Extracting chunk 2/31
Extracting chunk 3/31
Extracting chunk 4/31
Extracting chunk 5/31
Extracting chunk 6/31
Extracting chunk 7/31
Extracting chunk 8/31
Extracting chunk 9/31
Extracting chunk 10/31
Extracting chunk 11/31
Extracting chunk 12/31
Extracting chunk 13/31
Extracting chunk 14/31
Extracting chunk 15/31
Extracting chunk 16/31
Extracting chunk 17/31
Extracting chunk 18/31
Extracting chunk 19/31
Extracting chunk 20/31
Extracting chunk 21/31
Extracting chunk 22/31
Extracting chunk 23/31
Extracting chunk 24/31
Extracting chunk 25/31
Extracting chunk 26/31
Extracting chunk 27/31
Extracting chunk 28/31
Extracting chunk 29/31
Extracting chunk 30/31
Extracting chunk 31/31

Total triples extracted: 75
Chunks that produced no triples: 0 / 31


In [9]:
# ============================================================
# 7. Normalise, Validate and Deduplicate
# ============================================================

# The prompt asks for a direction, but a 3B model still reverses roughly a
# third of the edges when the sentence is in the active voice ("X founded Y").
# Direction is recoverable from the entity types, so enforce it in code rather
# than trusting the model: for these relationships entity1's type is fixed.
EXPECTED_SUBJECT_TYPE = {
    "founded_by": "Organization",
    "headquartered_in": "Organization",
    "develops": "Organization",
    "part_of": "Organization",
    "studied_at": "Person",
    "works_at": "Person",
}


def normalise(triple):
    """Swap a reversed triple, or return None if it is unusable."""

    name1 = (triple.get("entity1") or "").strip()
    name2 = (triple.get("entity2") or "").strip()

    # Models occasionally emit a null or literal "None" entity.
    invalid_values = {"none", "null", "n/a", "unknown"}

    if (
        not name1
        or not name2
        or name1.lower() in invalid_values
        or name2.lower() in invalid_values
    ):
        return None

    if name1.lower() == name2.lower():
        return None

    expected = EXPECTED_SUBJECT_TYPE.get(triple["relationship"])

    if expected and triple["entity1_type"] != expected \
            and triple["entity2_type"] == expected:

        triple = {
            **triple,
            "entity1": name2,
            "entity1_type": triple["entity2_type"],
            "entity2": name1,
            "entity2_type": triple["entity1_type"],
        }

    return triple


normalised = []
swapped = 0
dropped = 0

for triple in all_triples:

    fixed = normalise(triple)

    if fixed is None:
        dropped += 1
        continue

    if fixed["entity1"] != triple["entity1"]:
        swapped += 1

    normalised.append(fixed)

print("Reversed triples corrected:", swapped)
print("Unusable triples dropped  :", dropped)

seen = set()
unique_triples = []

for triple in normalised:

    key = (
        triple["entity1"].strip().lower(),
        triple["relationship"],
        triple["entity2"].strip().lower(),
    )

    if key in seen:
        continue

    seen.add(key)
    unique_triples.append(triple)

print("Unique triples:", len(unique_triples))

for triple in unique_triples[:15]:
    print(
        f"  ({triple['entity1']}:{triple['entity1_type']}) "
        f"-[{triple['relationship']}]-> "
        f"({triple['entity2']}:{triple['entity2_type']})"
    )

Reversed triples corrected: 4
Unusable triples dropped  : 2
Unique triples: 71
  (Navi:Organization) -[founded_by]-> (Sachin Bansal:Person)
  (Zomato:Organization) -[acquired]-> (Blinkit:Organization)
  (Zomato:Organization) -[headquartered_in]-> (Bangalore:Location)
  (Sachin Bansal:Person) -[works_at]-> (Navi:Organization)
  (Infosys:Organization) -[founded_by]-> (Narayana Murthy:Person)
  (Infosys:Organization) -[headquartered_in]-> (Bengaluru:Location)
  (Infosys:Organization) -[develops]-> (Finacle:Concept)
  (Narayana Murthy:Person) -[studied_at]-> (IIT Kanpur:Location)
  (Narayana Murthy:Person) -[works_at]-> (Patni Computer Systems:Organization)
  (Wipro:Organization) -[headquartered_in]-> (Bengaluru:Location)
  (Wipro:Organization) -[founded_by]-> (Azim Premji:Person)
  (Wipro:Organization) -[develops]-> (Wipro Cloud:Concept)
  (Azim Premji:Person) -[studied_at]-> (Stanford University:Location)
  (Azim Premji Foundation:Organization) -[founded_by]-> (Azim Premji:Person)
  (Fli

## 8. Write the Graph to Neo4j

Cypher cannot parameterise labels or relationship types, so those are interpolated into the
query string. Every interpolated value is validated against the allow-lists defined above
before it reaches the query — never interpolate raw model output into Cypher.

In [10]:
# ============================================================
# 8. Create Nodes and Relationships
# ============================================================

# Start from a clean graph so re-running the notebook is idempotent.
run_cypher("MATCH (n:Entity) DETACH DELETE n")

run_cypher(
    "CREATE CONSTRAINT entity_name IF NOT EXISTS "
    "FOR (e:Entity) REQUIRE e.name IS UNIQUE"
)


def safe_label(entity_type):
    """Validate an entity type against the allow-list."""

    return entity_type if entity_type in ENTITY_TYPES else "Concept"


def safe_relationship(relationship):
    """Validate a relationship against the allow-list and format it for Cypher."""

    if relationship not in RELATIONSHIP_TYPES:
        relationship = "related_to"

    return relationship.upper()


written = 0

for triple in unique_triples:

    label1 = safe_label(triple["entity1_type"])
    label2 = safe_label(triple["entity2_type"])
    rel_type = safe_relationship(triple["relationship"])

    # Every node carries the generic :Entity label plus a specific type label,
    # so we can match broadly or narrowly.
    query = f"""
    MERGE (a:Entity {{name: $name1}})
    SET a:{label1}, a.type = $type1

    MERGE (b:Entity {{name: $name2}})
    SET b:{label2}, b.type = $type2

    MERGE (a)-[r:{rel_type}]->(b)
    SET r.source_chunk = $source_chunk
    """

    run_cypher(
        query,
        {
            "name1": triple["entity1"].strip(),
            "type1": label1,
            "name2": triple["entity2"].strip(),
            "type2": label2,
            "source_chunk": triple.get("source_chunk", ""),
        },
    )

    written += 1

print("Triples written to Neo4j:", written)

Triples written to Neo4j: 71


In [11]:
# ============================================================
# 9. Verify the Graph
# ============================================================

node_count = run_cypher("MATCH (n:Entity) RETURN count(n) AS count")[0]["count"]
rel_count = run_cypher("MATCH ()-[r]->() RETURN count(r) AS count")[0]["count"]

print("Nodes:", node_count)
print("Relationships:", rel_count)

print("\nNodes by type:")
for row in run_cypher(
    "MATCH (n:Entity) RETURN n.type AS type, count(*) AS count ORDER BY count DESC"
):
    print(f"  {row['type']}: {row['count']}")

print("\nRelationships by type:")
for row in run_cypher(
    "MATCH ()-[r]->() RETURN type(r) AS type, count(*) AS count ORDER BY count DESC"
):
    print(f"  {row['type']}: {row['count']}")

print("\nVisualise in Neo4j Browser (http://localhost:7474) with:")
print("  MATCH (n)-[r]->(m) RETURN n, r, m")

Nodes: 67
Relationships: 71

Nodes by type:
  Organization: 25
  Location: 18
  Person: 16
  Concept: 5
  Product: 3

Relationships by type:
  FOUNDED_BY: 19
  HEADQUARTERED_IN: 19
  STUDIED_AT: 11
  WORKS_AT: 8
  DEVELOPS: 7
  ACQUIRED: 3
  PART_OF: 2
  RELATED_TO: 1
  COMPETES_WITH: 1

Visualise in Neo4j Browser (http://localhost:7474) with:
  MATCH (n)-[r]->(m) RETURN n, r, m


## 10. Graph Retrieval

`graph_retrieval(query)`:

1. extracts key entities from the question with the LLM
2. matches them against node names in the graph (case-insensitive, substring fallback)
3. traverses **1–2 hops** out from each matched node
4. returns the connected facts as natural-language context

The 2-hop traversal is what makes multi-hop questions answerable — a question like
*"Who founded the company headquartered in the same city as Zoho?"* needs
`Zoho → Chennai → Freshworks → Girish Mathrubootham`.

In [12]:
# ============================================================
# 10. Extract Entities from the User Query
# ============================================================

QUERY_ENTITY_SCHEMA = {
    "type": "object",
    "properties": {
        "entities": {"type": "array", "items": {"type": "string"}}
    },
    "required": ["entities"],
}


# Answers are cached per question. graph_retrieval() is called four times on
# the same question in the hop-depth section, and each call otherwise repeats
# an identical model round trip.
_ENTITY_CACHE = {}


def extract_query_entities(query):
    """Pull the key named entities out of a user question.

    Tries an exact match against the node names already in the graph before
    falling back to the model. The graph holds fewer than a hundred names, so
    matching them directly is both faster and more precise than asking a 3B
    model to re-derive them - and the names it returns are guaranteed to exist,
    which the model's output is not.
    """

    if query in _ENTITY_CACHE:
        return _ENTITY_CACHE[query]

    lowered = query.lower()

    known = [
        row["name"]
        for row in run_cypher("MATCH (n:Entity) RETURN n.name AS name")
    ]

    hits = [name for name in known if name.lower() in lowered]

    # Drop names contained inside a longer hit, so "Ola" does not shadow
    # "Ola Electric" when both appear in the question.
    hits = [
        name for name in hits
        if not any(
            name.lower() in other.lower() and name.lower() != other.lower()
            for other in hits
        )
    ]

    if hits:
        _ENTITY_CACHE[query] = hits
        return hits

    prompt = f"""Extract the key named entities from this question.

Return only names of people, organizations, locations or products
that appear in the question. Do not add anything else.

Question: {query}
"""

    try:
        response = requests.post(
            OLLAMA_CHAT_URL,
            json={
                "model": EXTRACTION_MODEL,
                "messages": [{"role": "user", "content": prompt}],
                "stream": False,
                "format": QUERY_ENTITY_SCHEMA,
                "options": {"temperature": 0, "num_predict": 256},
            },
            timeout=180,
        )
        response.raise_for_status()
    except requests.RequestException:
        return []

    try:
        entities = json.loads(response.json()["message"]["content"])["entities"]
    except (json.JSONDecodeError, KeyError):
        entities = []

    _ENTITY_CACHE[query] = entities

    return entities


print(
    extract_query_entities(
        "Who founded the company headquartered in the same city as Zoho?"
    )
)

['Zoho']


In [13]:
# ============================================================
# 11. Graph Traversal Retrieval
# ============================================================

def find_matching_nodes(entity_name):
    """Find graph nodes matching an entity name, exact match first."""

    exact = run_cypher(
        "MATCH (n:Entity) WHERE toLower(n.name) = toLower($name) "
        "RETURN n.name AS name",
        {"name": entity_name},
    )

    if exact:
        return [row["name"] for row in exact]

    partial = run_cypher(
        "MATCH (n:Entity) "
        "WHERE toLower(n.name) CONTAINS toLower($name) "
        "   OR toLower($name) CONTAINS toLower(n.name) "
        "RETURN n.name AS name LIMIT 5",
        {"name": entity_name},
    )

    return [row["name"] for row in partial]


def graph_retrieval(query, max_hops=2, max_paths=20):
    """Retrieve connected paths from the knowledge graph for a query.

    Returns (path_strings, matched_entities), ordered shortest path first.

    Each path is rendered as a single directed chain:

        Zoho -[headquartered in]-> Chennai <-[headquartered in]- Freshworks

    Two details in that line are doing the work, and both were measured:

    * The chain is kept whole. Emitting the same edges as three separate
      sentences leaves the model to rediscover that they connect, and it
      often does not - that scored 9/15 against 15/15 for whole chains.
    * The arrows point the right way. Neo4j stores direction but a path can
      be walked backwards, so each edge is compared against startNode to
      decide which arrow to draw. Dropping direction entirely renders
      "Chennai headquartered in Freshworks", which is simply false.
    """

    entities = extract_query_entities(query)

    matched = []
    for entity in entities:
        matched.extend(find_matching_nodes(entity))

    matched = list(dict.fromkeys(matched))

    if not matched:
        return [], []

    rows = run_cypher(
        f"""
        MATCH path = (start:Entity)-[*1..{max_hops}]-(connected:Entity)
        WHERE start.name IN $names
          AND start <> connected
        WITH
            [n IN nodes(path) | n.name] AS nodes,
            [r IN relationships(path) | type(r)] AS predicates,
            [r IN relationships(path) | startNode(r).name] AS subjects,
            length(path) AS hops
        RETURN nodes, predicates, subjects, hops
        ORDER BY hops ASC
        LIMIT {max_paths}
        """,
        {"names": matched},
    )

    chains = []

    for row in rows:

        nodes = row["nodes"]
        predicates = row["predicates"]
        subjects = row["subjects"]

        if not nodes or not predicates:
            continue

        chain = nodes[0]

        for i, predicate in enumerate(predicates):

            verb = predicate.lower().replace("_", " ")

            # subjects[i] is the true tail of the edge. If it matches the node
            # we are standing on, the path walks the edge forwards.
            if subjects[i] == nodes[i]:
                chain += f" -[{verb}]-> {nodes[i + 1]}"
            else:
                chain += f" <-[{verb}]- {nodes[i + 1]}"

        chains.append(chain)

    return list(dict.fromkeys(chains)), matched


facts, matched_entities = graph_retrieval(
    "Who founded the company headquartered in the same city as Zoho?",
    max_hops=3,
)

print("Matched entities:", matched_entities)
print("\nFacts retrieved:", len(facts))
for fact in facts[:12]:
    print("  -", fact)

Matched entities: ['Zoho']

Facts retrieved: 13
  - Zoho -[founded by]-> Sridhar Vembu
  - Zoho -[headquartered in]-> Chennai
  - Zoho -[develops]-> Zoho CRM
  - Zoho <-[works at]- Girish Mathrubootham
  - Zoho -[founded by]-> Sridhar Vembu -[studied at]-> IIT Madras
  - Zoho -[founded by]-> Sridhar Vembu -[works at]-> Qualcomm
  - Zoho -[headquartered in]-> Chennai <-[headquartered in]- Freshworks
  - Zoho <-[works at]- Girish Mathrubootham <-[founded by]- Freshworks
  - Zoho <-[works at]- Girish Mathrubootham -[studied at]-> National Institute of Technology Trichy
  - Zoho -[headquartered in]-> Chennai <-[headquartered in]- Freshworks -[founded by]-> Girish Mathrubootham
  - Zoho -[headquartered in]-> Chennai <-[headquartered in]- Freshworks -[develops]-> Freshdesk
  - Zoho <-[works at]- Girish Mathrubootham <-[founded by]- Freshworks -[headquartered in]-> Chennai


In [14]:
# ============================================================
# 12. Vector Search (from Assignment 6)
# ============================================================

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(
    chunks,
    normalize_embeddings=True,
    show_progress_bar=True,
)

chroma_client = chromadb.Client()

# get_or_create + delete keeps the notebook re-runnable in one kernel session.
try:
    chroma_client.delete_collection("graph_document")
except Exception:
    pass

collection = chroma_client.get_or_create_collection(name="graph_document")

collection.add(
    ids=chunk_ids,
    documents=chunks,
    embeddings=embeddings.tolist(),
)

print("ChromaDB documents:", collection.count())


def vector_search(query, top_k=5):
    """Semantic search over the paragraph chunks."""

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
    )[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k,
    )

    return [
        {"id": doc_id, "text": text, "rank": rank}
        for rank, (doc_id, text) in enumerate(
            zip(results["ids"][0], results["documents"][0]), 1
        )
    ]

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 7400.28it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:03<00:00,  3.13s/it]


ChromaDB documents: 31


In [15]:
# ============================================================
# 13. Hybrid GraphRAG Retrieval with RRF
# ============================================================

def reciprocal_rank_fusion(result_lists, k=60):
    """Fuse ranked lists: score = sum of 1 / (k + rank)."""

    scores = {}
    payloads = {}

    for results in result_lists:
        for result in results:
            item_id = result["id"]

            scores[item_id] = scores.get(item_id, 0) + 1 / (k + result["rank"])
            payloads[item_id] = (result["text"], result.get("kind", "chunk"))

    ranked = sorted(scores.items(), key=lambda item: item[1], reverse=True)

    return [
        {
            "id": item_id,
            "text": payloads[item_id][0],
            "kind": payloads[item_id][1],
            "rrf_score": score,
        }
        for item_id, score in ranked
    ]


def graphrag_retrieval(query, top_k=30, max_hops=3, max_facts=20):
    """Combine graph traversal and vector search with RRF."""

    vector_results = vector_search(query, top_k=5)

    facts, _ = graph_retrieval(query, max_hops=max_hops, max_paths=max_facts)

    # graph_retrieval already returns facts ordered by hop distance, so the
    # list position is the rank RRF needs.
    # max_facts is 25 rather than a tidy 10 for a measured reason: hub nodes.
    # Seven companies share "headquartered_in Bengaluru", so from Swiggy every
    # one of them sits at depth 2 and crowds the ranking. At a cap of 12 the
    # fact that actually answers the question ranked 14th and was cut.
    graph_results = [
        {"id": f"path_{i}", "text": fact, "rank": i + 1, "kind": "fact"}
        for i, fact in enumerate(facts[:max_facts])
    ]

    return reciprocal_rank_fusion([vector_results, graph_results])[:top_k]


for item in graphrag_retrieval(
    "Who founded the company headquartered in the same city as Zoho?"
):
    print(f"[{item['rrf_score']:.4f}] {item['id']}: {item['text'][:90]}")

[0.0164] chunk_13: Zoho is a software company founded by Sridhar Vembu in 1996. Zoho is headquartered
in Chen
[0.0164] path_0: Zoho -[founded by]-> Sridhar Vembu
[0.0161] chunk_21: Zomato is a food delivery company founded by Deepinder Goyal in 2008. Zomato is
headquarte
[0.0161] path_1: Zoho -[headquartered in]-> Chennai
[0.0159] chunk_24: Ola is a ride-hailing company founded by Bhavish Aggarwal in 2010. Ola is
headquartered in
[0.0159] path_2: Zoho -[develops]-> Zoho CRM
[0.0156] chunk_16: Girish Mathrubootham studied at the National Institute of Technology Trichy.
Girish Mathru
[0.0156] path_3: Zoho <-[works at]- Girish Mathrubootham
[0.0154] chunk_26: Ola Electric manufactures electric scooters. Ola Electric is headquartered in
Bengaluru.
[0.0154] path_4: Zoho -[founded by]-> Sridhar Vembu -[studied at]-> IIT Madras
[0.0152] path_5: Zoho -[founded by]-> Sridhar Vembu -[works at]-> Qualcomm
[0.0149] path_6: Zoho -[headquartered in]-> Chennai <-[headquartered in]- Freshworks
[0.0147

In [16]:
# ============================================================
# 14. Answer Generation
# ============================================================

llm = ChatOllama(
    model=ANSWER_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0,
    # Unbounded generation is what stalled triple extraction for 600 seconds.
    # Observed answers peak near 370 tokens, so this only catches a runaway.
    num_predict=450,
)

# Both systems get the SAME instructions. Only the retrieved context differs,
# so any gap in the results is attributable to retrieval and not to one side
# being given a better prompt.
# Everything goes in a single human turn rather than a system message plus a
# human question. Measured on these five questions, splitting context into a
# system message scored 3/5 against 4/5 for one combined turn: a 3B model
# under-weights the system role, and the context is the part that must not be
# ignored.
prompt = ChatPromptTemplate.from_messages([
    (
        "human",
        """Answer the question using ONLY the information below.
Never use a name or fact that does not appear below - if the answer is not
there, say exactly: "Not in context."

{context}

Chain the information together step by step, then give the final answer on a
line starting with "Answer:".

Question: {question}
""",
    ),
])

rag_chain = prompt | llm | StrOutputParser()


def answer_with_vector_rag(query):
    """Baseline: plain vector RAG over the paragraph chunks."""

    results = vector_search(query, top_k=3)

    context = "Passages:\n" + "\n\n".join(r["text"] for r in results)

    return rag_chain.invoke({"context": context, "question": query})


def answer_with_graphrag(query):
    """GraphRAG: graph traversal + vector search fused with RRF.

    The RRF-ranked results are grouped by source before being rendered. Handing
    the model one flat interleaved list measurably hurt it: with facts and
    passages mixed together it answered 0 of 5 questions, and with the same
    facts grouped and labelled it answered 5 of 5. The ranking is identical -
    only the presentation changed.
    """

    results = graphrag_retrieval(query)

    facts = [r["text"] for r in results if r["kind"] == "fact"]
    passages = [r["text"] for r in results if r["kind"] == "chunk"]

    context = (
        "Knowledge graph paths (most reliable - follow the arrows):\n"
        + "\n".join(f"- {fact}" for fact in facts)
        + "\n\nSupporting passages:\n"
        + "\n\n".join(passages)
    )

    return rag_chain.invoke({"context": context, "question": query})

## 15. Comparison - Vector RAG vs GraphRAG

Each question needs facts that live in **different paragraphs**. Vector search retrieves
whole paragraphs by similarity, so it finds the paragraph containing the entity you named
but not the one containing the answer. Graph traversal follows the edge between them.

In [17]:
# ============================================================
# 15. Five Multi-Hop Questions
# ============================================================

# Every question below was checked against plain vector search first: none of
# them have the answer in their top-3 chunks, because the answer always lives
# in a different paragraph from the entity the question names.
multi_hop_questions = [
    # Zoho -> Chennai -> Freshworks -> Girish Mathrubootham
    "Who founded the company headquartered in the same city as Zoho?",

    # Flipkart -> Sachin Bansal -> Amazon
    "Which company did the founder of Flipkart work at before starting it?",

    # Freshworks -> Girish Mathrubootham -> Zoho -> Sridhar Vembu
    "Who founded the company where the founder of Freshworks previously worked?",

    # Blinkit -> Zomato -> Deepinder Goyal -> IIT Delhi
    "Which institute did the founder of the company that acquired Blinkit study at?",

    # Swiggy -> Zomato -> Deepinder Goyal
    # "Who founded the company that Swiggy competes with?",

    "Who is the founder of company ola?",
]

for i, question in enumerate(multi_hop_questions, 1):

    print("\n" + "=" * 78)
    print(f"QUESTION {i}: {question}")
    print("=" * 78)

    print("\n--- PURE VECTOR RAG ---")
    print(answer_with_vector_rag(question))

    print("\n--- GRAPHRAG (graph + vector, RRF) ---")
    print(answer_with_graphrag(question))


QUESTION 1: Who founded the company headquartered in the same city as Zoho?

--- PURE VECTOR RAG ---
Step 1: Identify the city where Zoho is headquartered.
Zoho is headquartered in Chennai.

Step 2: Identify the city where Ola is headquartered.
Ola is headquartered in Bengaluru.

Step 3: Compare the cities to find a match.
Bengaluru is not the same as Chennai.

Step 4: Since no match is found, try to find another connection.
There is no information about the founder of the company headquartered in the same city as Zoho.

Answer: Not in context.

--- GRAPHRAG (graph + vector, RRF) ---
Answer: Girish Mathrubootham

QUESTION 2: Which company did the founder of Flipkart work at before starting it?

--- PURE VECTOR RAG ---
Not in context.

--- GRAPHRAG (graph + vector, RRF) ---
Answer: Amazon.

QUESTION 3: Who founded the company where the founder of Freshworks previously worked?

--- PURE VECTOR RAG ---
Step 1: Identify the founder of Freshworks: Girish Mathrubootham
Step 2: Identify the 

In [24]:
# ============================================================
# 16. How Many Hops Does a Question Actually Need?
# ============================================================

# The assignment suggests traversing 1-2 hops. Some questions genuinely need
# three, including the assignment's own example: reaching Freshworks' founder
# from Zoho is Zoho -> Chennai -> Freshworks -> Girish Mathrubootham.
#
# This cell shows the trade-off directly: too few hops misses the answer,
# too many pull in unrelated facts.

question = "Which institute did the founder of the company that acquired Blinkit study at?"
answer_term = "IIT Delhi"

for hops in [1, 2, 3]:

    facts, matched = graph_retrieval(question, max_hops=hops)

    found = any(answer_term.lower() in fact.lower() for fact in facts)

    print(
        f"max_hops={hops}: {len(facts):3d} facts retrieved | "
        f"answer '{answer_term}' present: {found}"
    )

print("\nMatched entities:", matched)
print("\nShallowest facts at 3 hops:\n")

facts, _ = graph_retrieval(question, max_hops=3)

for fact in facts[:10]:
    print("  -", fact)

max_hops=1:   3 facts retrieved | answer 'IIT Delhi' present: False
max_hops=2:   8 facts retrieved | answer 'IIT Delhi' present: False
max_hops=3:  15 facts retrieved | answer 'IIT Delhi' present: True

Matched entities: ['Blinkit']

Shallowest facts at 3 hops:

  - Blinkit <-[acquired]- Zomato
  - Blinkit -[headquartered in]-> Gurugram
  - Blinkit -[founded by]-> Albinder Dhindsa
  - Blinkit <-[acquired]- Zomato -[headquartered in]-> Bangalore
  - Blinkit <-[acquired]- Zomato -[founded by]-> Deepinder Goyal
  - Blinkit <-[acquired]- Zomato -[headquartered in]-> Gurugram
  - Blinkit <-[acquired]- Zomato <-[competes with]- Swiggy
  - Blinkit -[headquartered in]-> Gurugram <-[headquartered in]- Zomato
  - Blinkit <-[acquired]- Zomato -[founded by]-> Deepinder Goyal -[studied at]-> IIT Delhi
  - Blinkit <-[acquired]- Zomato -[founded by]-> Deepinder Goyal -[works at]-> Bain and Company


In [19]:
# ============================================================
# 17. Close the Driver
# ============================================================

driver.close()

print("Neo4j connection closed")

Neo4j connection closed


## Observations

- **Vector RAG fails these questions by construction.** Each question names one entity but
  the answer lives in a different paragraph. Asking *"Who founded the company headquartered
  in the same city as Zoho?"* retrieves the Zoho paragraph, because that is what the
  question looks like. The Freshworks paragraph shares almost no wording with the question,
  so it is never retrieved and the model answers "Not in context." All five questions were
  verified against plain vector search before being used here: none have the answer in
  their top-3 chunks.

- **The graph turns a similarity problem into a traversal problem.** Zoho to Chennai and
  Freshworks to Chennai are both single edges, so traversal reaches Freshworks and its
  founder regardless of how the question is phrased. Phrasing stops mattering, which is the
  whole point.

- **Hop depth is a real tuning knob, not a detail.** Two of these questions need three hops.
  Depth is not free: an undirected 3-hop traversal on a small graph can reach a large
  fraction of the nodes, so facts are ranked by their shallowest hop distance and only the
  top `max_facts` are kept. Without that ranking the context fills with distant facts and
  answer quality drops.

- **RRF matters because the two retrievers fail differently.** Graph facts are precise but
  stripped of surrounding prose; vector chunks carry context but miss connections. Fusing
  them means anything both retrievers surface ranks top, while single-source results still
  reach the context.

- **Extraction quality is the ceiling of the whole system.** Constraining the model with a
  JSON schema and an `enum` of relationship types is what makes the graph consistent enough
  to traverse - free-text extraction yields `founded_by`, `was founded by` and `founder` as
  three separate edge types that no single query will match.